**Title:** Multi-Agent Content Pipeline — Writer, Editor & SEO Specialist Agents  
**Description:** Built a sequential multi-agent team where a supervisor coordinates three specialist agents (`writer`, `editor`, `SEO` `optimizer`) in a fixed pipeline order, testing sequential collaboration instead of parallel routing.

In [1]:
import os
import time
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

**check env**

In [2]:
load_dotenv()
print("env loaded:", "GEMINI_API_KEY" in os.environ)

env loaded: True


**Setup llm**

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

**Worker 1: Draft Writer Agent**

In [4]:
writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""You are a draft writer. Given a topic, write a clear, engaging short draft (3-4 paragraphs).
Do not worry about grammar perfection or SEO — focus on getting the core ideas down well."""
)

**Worker 2: Editor Agent**

In [5]:
editor_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""You are a professional editor. You will be given a draft.
Improve clarity, fix grammar, tighten sentences, and improve flow.
Do NOT change the core meaning or add new facts. Return only the edited version."""
)

**Worker 3: SEO Optimizer Agent**

In [6]:
seo_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""You are an SEO specialist. You will be given an edited article.
Suggest a compelling title, a meta description (under 160 characters), and 5 relevant keywords.
Do not rewrite the article body — only provide the title, meta description, and keywords."""
)

**Note:** these three workers have no tools, they're pure LLM specialists, not tool-users.  multi-agent systems don't require every worker to have external tools; specialization can come purely from a focused system prompt and role.

**Wrap each worker as a callable tool for the supervisor**

In [7]:
@tool
def write_draft(topic: str) -> str:
    """Generates a first draft article on the given topic. Always call this FIRST in the pipeline."""
    result = writer_agent.invoke({"messages": [{"role": "user", "content": f"Write a short draft about: {topic}"}]})
    return result["messages"][-1].content

@tool
def edit_draft(draft_text: str) -> str:
    """Edits and polishes a draft for clarity and grammar. Call this SECOND, after write_draft."""
    result = editor_agent.invoke({"messages": [{"role": "user", "content": f"Edit this draft:\n\n{draft_text}"}]})
    return result["messages"][-1].content

@tool
def optimize_seo(edited_text: str) -> str:
    """Generates title, meta description, and keywords for an edited article. Call this THIRD and LAST, after edit_draft."""
    result = seo_agent.invoke({"messages": [{"role": "user", "content": f"Provide SEO elements for this article:\n\n{edited_text}"}]})
    return result["messages"][-1].content

**Build the Supervisor Agent — enforces sequential pipeline order**

In [8]:
supervisor_prompt = """You are a content production supervisor managing three specialists: write_draft, edit_draft, and optimize_seo.

Rules:
1. ALWAYS follow this exact order: write_draft FIRST, then edit_draft on the result, then optimize_seo on the edited result. Never skip a step. Never reorder.
2. Pass the ACTUAL output of one tool as the input to the next tool — do not summarize or shorten it yourself in between steps.
3. Your final answer must include: the edited article text, followed by the SEO title, meta description, and keywords.
4. Do not write or edit content yourself — always delegate to the appropriate specialist tool.
5. Generate your final answer only ONCE."""

supervisor_agent = create_agent(
    model=llm,
    tools=[write_draft, edit_draft, optimize_seo],
    system_prompt=supervisor_prompt
)

**Retry wrapper for rate limits**

In [9]:
def invoke_agent_safely(agent, messages, max_retries=2, base_delay=10):
    for attempt in range(1, max_retries + 1):
        try:
            return agent.invoke({"messages": messages})
        except Exception as e:
            if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < max_retries:
                print(f"Rate limit hit. Waiting {base_delay}s...")
                time.sleep(base_delay)
            else:
                print(f"Failed: {e}")
                return None

**Run the full pipeline**

In [10]:
result = invoke_agent_safely(
    supervisor_agent,
    [{"role": "user", "content": "Create content about the benefits of learning Python for beginners."}]
)

if result:
    for msg in result["messages"]:
        print(f"[{msg.type}] {msg.content}")

[human] Create content about the benefits of learning Python for beginners.
[ai] []
[tool] [{'type': 'text', 'text': "Stepping into the world of programming can feel overwhelming, but Python has quickly become the go-to language for anyone starting out. Unlike older, more rigid languages, Python was designed with human readability in mind. Its syntax closely resembles plain English, which means beginners don't have to get bogged down by complex punctuation and cryptic commands right away. Instead of spending hours figuring out why a semicolon is missing, new coders can focus immediately on learning how to think logically and solve actual problems.\n\nBeyond its gentle learning curve, Python is remarkably versatile. Whether you are interested in building websites, analyzing massive datasets, automating boring daily tasks, or even exploring artificial intelligence, Python can do it all. This incredible flexibility means that as your interests evolve, you don't need to learn a whole new l

**Check the pipeline order was actually followed**

In [11]:
if result:
    tool_msgs = [m for m in result["messages"] if m.type == "tool"]
    print(f"Total tool calls: {len(tool_msgs)}")
    for i, m in enumerate(tool_msgs, 1):
        print(f"\nStep {i} output (first 150 chars):\n{m.content[:150]}")

Total tool calls: 3

Step 1 output (first 150 chars):
[{'type': 'text', 'text': "Stepping into the world of programming can feel overwhelming, but Python has quickly become the go-to language for anyone starting out. Unlike older, more rigid languages, Python was designed with human readability in mind. Its syntax closely resembles plain English, which means beginners don't have to get bogged down by complex punctuation and cryptic commands right away. Instead of spending hours figuring out why a semicolon is missing, new coders can focus immediately on learning how to think logically and solve actual problems.\n\nBeyond its gentle learning curve, Python is remarkably versatile. Whether you are interested in building websites, analyzing massive datasets, automating boring daily tasks, or even exploring artificial intelligence, Python can do it all. This incredible flexibility means that as your interests evolve, you don't need to learn a whole new language from scratch. You simply use 